**Roadmap**

*Questions*<br>
How drastically has the average land surface temperature changed since the mid-1980s?<br>
How does temperature correlate with vegetation?<br>
How do spectral indices help us interpret the physical causes of these heat patterns?<br>

*Data*<br>
Needed: Temperature (LST) and Vegetation (NDVI)<br>
Imported: Landsat imagery (1985-2024, 3-yearly)<br>

*Output*<br>
LST and NDVI Maps (snapshots and change over time)<br>
Statistical analysis of LST and NDVI<br>

**Data Import**

Initial import of Landsat dataset into data cube<br>
Calculation of NDVI and conversion of LST from K to °C<br>

In [ ]:
#
# Import Landsat Data and Calculate NDVI
#

import rasterio
import os
import geopandas as gpd
import xarray as xr
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import gdown
from scipy.stats import linregress
import cmcrameri.cm as cmc
from matplotlib.colors import LightSource

# Loop over years to create filepaths filepaths
years = [1985, 1988, 1991, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024]

ls_paths = []

for year in years:
	fp = "data/raw/LandsatComposite_Zurich_" + str(year) + ".tif"

	ls_paths.append(fp)

# Loop over filepaths to import data into data cube
list_ds = []

for path in ls_paths:
	da = rioxarray.open_rasterio(path)

	dt = int(path.split("_")[2].split(".")[0])

	ds = xr.Dataset(
		{
			"Blue": da.sel(band = 1).drop_vars("band", errors="ignore"), 
			"Green": da.sel(band = 2).drop_vars("band", errors="ignore"), 
			"Red": da.sel(band = 3).drop_vars("band", errors="ignore"), 
			"NIR": da.sel(band = 4).drop_vars("band", errors="ignore"), 
			"SWIR1":  da.sel(band = 5).drop_vars("band", errors="ignore"), 
			"SWIR2": da.sel(band = 6).drop_vars("band", errors="ignore"), 
			"TIR": da.sel(band = 7).drop_vars("band", errors="ignore"),
		}
	)

	#ds = ds.assign_coords(band = bands)
	ds = ds.expand_dims(time = [dt])
	ds["NDVI"] = (ds.NIR - ds.Red) / (ds.NIR + ds.Red)
	ds["LST"] = ds.TIR - 273.15

	list_ds.append(ds)

ls_cube = xr.combine_by_coords(list_ds)

#ls_cube

**Visualisation of LST and NDVI in 2024**

2 maps, one showing LST and the other NDVI in 2024

In [ ]:
#
# Visualisation of Temperature and NDVI
#

# Temperature

lst_1985 = ls_cube.LST.sel(time=1985)
lst_2024 = ls_cube.LST.sel(time=2024)

fig, ax = plt.subplots()

lst_2024.plot(
	ax=ax, 
	cmap="RdBu_r", 
	#vmin = -5,
	#vmax = 5,
	cbar_kwargs={"label": "Temperature (°C)"}
)

ax.set_title("Land Surface Temperature (2024)")
plt.axis("off")
plt.tight_layout()
plt.show()

# NDVI

ndvi_1985 = ls_cube.NDVI.sel(time=1985)
ndvi_2024 = ls_cube.NDVI.sel(time=2024)

fig, ax = plt.subplots()

ndvi_2024.plot(
	ax=ax, 
	cmap="RdBu", 
	vmin = 0,
	vmax = 1,
	cbar_kwargs={"label": "NDVI"}
)

ax.set_title("NDVI (2024)")
plt.axis("off")
plt.tight_layout()
plt.show()

**Difference Maps for LST and NDVI**

Maps showing the change between 2024 and 1985

In [ ]:
#
# Difference 2024 to 1985
#

# LST Difference

lst_1985 = ls_cube.LST.sel(time=1985)
lst_2024 = ls_cube.LST.sel(time=2024)

lst_diff = lst_2024 - lst_1985

fig, ax = plt.subplots()

lst_diff.plot(
	ax=ax, 
	cmap="RdBu_r", 
	vmin = -5,
	vmax = 5,
	cbar_kwargs={"label": "Temperature Change (°C)"}
)

ax.set_title("Temperature Difference (1985-2024)")
plt.axis("off")
plt.tight_layout()
plt.show()

# NDVI Difference

ndvi_1985 = ls_cube.NDVI.sel(time=1985)
ndvi_2024 = ls_cube.NDVI.sel(time=2024)

ndvi_diff = ndvi_2024 - ndvi_1985

fig, ax = plt.subplots()

ndvi_diff.plot(
	ax=ax, 
	cmap="RdBu_r", 
	vmin = -1,
	vmax = 1,
	cbar_kwargs={"label": "NDVI Change"}
)

ax.set_title("Vegetation Difference (1985-2024)")
plt.axis("off")
plt.tight_layout()
plt.show()


**Trends of Full Dataset**

Linear regression across all available years for both LST and NDVI as a decadal trend<br>
Trend plotted as map for visual analysis

In [ ]:
#
# Decadal Trends
#

# Temperature

# Apply linear regression
trends = ls_cube.polyfit(dim="time", deg=1)

# Scale slope
trend_map = trends.LST_polyfit_coefficients.sel(degree=1) * 10

# Plot the trend map
fig, ax = plt.subplots()

trend_map.plot(
	ax=ax, 
	cmap='RdBu_r', 
	cbar_kwargs={'label': 'Temperature Change per Decade (°C)'}
)

ax.set_title("Decadal Temperature Trend (1985-2024)")
plt.axis("off")
plt.tight_layout()
plt.show()

# NDVI

# Apply linear regression
trends = ls_cube.polyfit(dim="time", deg=1)

# Scale slope
trend_map = trends.NDVI_polyfit_coefficients.sel(degree=1) * 10

# Plot the trend map
fig, ax = plt.subplots()

trend_map.plot(
	ax=ax, 
	cmap='RdBu', 
	cbar_kwargs={'label': 'NDVI Change per Decade'}
)

ax.set_title("Decadal Trend of NDVI")
plt.axis("off")
plt.tight_layout()
plt.show()

**Analysis of 3-yearly Means**

Calculation of mean for each available image<br>
Line chart showing said means over time

In [ ]:
#
# 3-yearly Means
#

ls_mean = ls_cube.mean(dim=["x", "y"])

# Plot mean LST

ls_mean.LST.sel().plot(marker="o", figsize=(10, 4))
plt.title("3-yearly mean Temperature (1985-2024)")
plt.ylabel("Temperature (°C)")
plt.xlabel("Year")
plt.xticks(years)
plt.grid(alpha=0.3)
plt.show()

# Plot mean NDVI
ls_mean.NDVI.sel().plot(marker="o", figsize=(10, 4))
plt.title("3-yearly mean NDVI (1985-2024)")
plt.ylabel("NDVI")
plt.xlabel("Year")
plt.xticks(years)
plt.grid(alpha=0.3)
plt.show()

**Correlation of NDVI and LST**

Calculation of Pearson's correlation coefficient between NDVI and LST

In [ ]:
#
# Correlation of NDVI and LST
#

xr.corr(ls_cube.NDVI, ls_cube.LST)